# BF16 top-K

Each row is one `(tokens, experts, K)` case. **BF16 is the focus**; old FP32 and Torch are timing baselines. Speedup is `baseline time / BF16 time`: **above 1× favors BF16**. A dash means the old FP32 kernel does not support that case.

The accuracy table compares weights against Torch's FP32 scores at the selected expert IDs. It shows the **maximum relative weight error** for BF16 and old FP32. BF16 expert overlap measures agreement with Torch's chosen IDs; ties can differ. BF16 score loss is the largest per-token deficit in the sum of selected Torch scores.

All variants use the same BF16-rounded logits. Input conversion/allocation are outside CUDA-graph timing. Torch runs FP32 activation + top-K + output conversion. The BF16 correctness limits are 2% relative selection-score error and 5% weight error; tables show measured errors.

Run a checkout containing these changes; Colab clones `fmoe`. Restart the notebook kernel after rebuilding an extension already imported in this session.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

root = next((p for p in (Path.cwd(), Path.cwd().parent, Path.cwd() / "xCaliber")
             if (p / "xcaliber" / "setup.py").is_file()), None)
if root is None:
    root = Path.cwd() / "xCaliber"
    subprocess.run(["git", "clone", "-b", "fmoe", "https://github.com/Pranshu-Bahadur/xCaliber.git", str(root)], check=True)
root = root.resolve()
print(root)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "ninja", "pytest", "pandas", "jinja2"], check=True)


In [ ]:
import torch

assert torch.cuda.is_available(), "Select a CUDA GPU runtime"
os.environ["TORCH_CUDA_ARCH_LIST"] = ".".join(map(str, torch.cuda.get_device_capability()))
os.environ.setdefault("MAX_JOBS", "4")
print(torch.cuda.get_device_name(), "sm", os.environ["TORCH_CUDA_ARCH_LIST"], "CUDA", torch.version.cuda)
subprocess.run([sys.executable, str(root / "xcaliber" / "setup.py"), "build_ext", "--inplace"], cwd=root, check=True)


In [ ]:
subprocess.run([sys.executable, "-m", "pytest", str(root / "test" / "test_moe.py"), "-q"], cwd=root, check=True)


In [ ]:
import importlib.util

sys.path.insert(0, str(root))
spec = importlib.util.spec_from_file_location("test_moe", root / "test" / "test_moe.py")
moe_tests = importlib.util.module_from_spec(spec)
spec.loader.exec_module(moe_tests)


In [ ]:
print(torch.cuda.get_device_name(), "| Torch", torch.__version__, "| CUDA", torch.version.cuda)

Ns = (8, 16, 16384)
Es = (256, 512)
Ks = (2, 8)
results = moe_tests.benchmark(Ns=Ns, Es=Es, Ks=Ks, repeat=100, verbose=False)


In [ ]:
import pandas as pd
from IPython.display import display, Markdown

frame = pd.DataFrame(results)
focus = {"font-weight": "bold", "background-color": "#eaf3ff", "color": "#122d4a"}
for activation, group in frame.groupby("activation", sort=False):
    latency = group.pivot(index=["N", "E", "K"], columns="variant", values="us")
    latency = latency.reindex(columns=["bf16", "fp32", "torch"])
    latency["vs old FP32"] = latency["fp32"] / latency["bf16"]
    latency["vs Torch"] = latency["torch"] / latency["bf16"]
    latency = latency.rename(columns={"bf16": "BF16 (µs)", "fp32": "Old FP32 (µs)", "torch": "Torch (µs)"})
    latency = latency.rename_axis(index=["Tokens", "Experts", "K"], columns=None)
    display(Markdown(f"### {activation.title()} — latency & BF16 speedup"))
    display(latency.style.format({"BF16 (µs)": "{:.3f}", "Old FP32 (µs)": "{:.3f}", "Torch (µs)": "{:.3f}",
                                  "vs old FP32": "{:.2f}×", "vs Torch": "{:.2f}×"}, na_rep="—")
            .set_properties(subset=["BF16 (µs)", "vs old FP32", "vs Torch"], **focus))

    accuracy = group[group["variant"] == "bf16"].set_index(["N", "E", "K"])
    accuracy = accuracy[["max_rel_pct", "overlap_pct", "mass_gap"]].rename(columns={
        "max_rel_pct": "BF16 max error (%)", "overlap_pct": "BF16 expert overlap (%)", "mass_gap": "BF16 score loss"})
    accuracy["Old FP32 max error (%)"] = group[group["variant"] == "fp32"].set_index(["N", "E", "K"])["max_rel_pct"]
    accuracy = accuracy[["BF16 max error (%)", "Old FP32 max error (%)", "BF16 expert overlap (%)", "BF16 score loss"]]
    accuracy = accuracy.sort_index().rename_axis(index=["Tokens", "Experts", "K"])
    display(Markdown(f"### {activation.title()} — approximation vs Torch"))
    display(accuracy.style.format({"BF16 max error (%)": "{:.3f}", "Old FP32 max error (%)": "{:.3f}",
                                   "BF16 expert overlap (%)": "{:.3f}", "BF16 score loss": "{:.3e}"}, na_rep="—")
            .set_properties(subset=["BF16 max error (%)"], **focus))
